# 🛰️ Detección de Cambios en el Terreno — Área Metropolitana de Bucaramanga

**Detección de cambios en terrenos mediante el análisis de imágenes satelitales multitemporales**

Este notebook implementa, **fase a fase**, una herramienta para detectar cambios en la cobertura
terrestre (expansión urbana, pérdida/ganancia de vegetación, variaciones en cuerpos de agua) en el
**Área Metropolitana de Bucaramanga (AMB)** — Bucaramanga, Floridablanca, Girón y Piedecuesta —, a
partir de imágenes satelitales de acceso abierto, procesadas con **Google Earth Engine (GEE)**.

## Objetivo general
Desarrollar una herramienta para la detección de cambios en la cobertura terrestre mediante el
análisis de imágenes satelitales multitemporales, aplicando técnicas de procesamiento digital de
imágenes, con la finalidad de mejorar la eficiencia del monitoreo ambiental en un área específica.

## Alcance temporal y fuentes de datos

Se analiza el periodo **2010-2026** (17 años), el rango completo solicitado para el estudio. Ninguna
plataforma óptica de acceso abierto cubre por sí sola todo ese rango con buena resolución, así que se
usan **dos familias de sensores** según el año, documentado con detalle en la Fase 1.4:

| Años | Sensor | Resolución | Motivo |
|---|---|---|---|
| 2010-2011 | Landsat 5 TM | 30 m | Sentinel-2 no existe aún; Landsat 5 es la plataforma disponible. |
| 2012 | Landsat 7 ETM+ | 30 m | Año puente: Landsat 5 ya no opera y Landsat 8 aún no se ha lanzado. |
| 2013-2016 | Landsat 8 OLI | 30 m | Sentinel-2 aún no tiene cobertura operativa consistente sobre Colombia. |
| 2017-2026 | Sentinel-2 SR | 10 m | Mayor detalle espacial una vez la cobertura es consistente. |

Cada año se representa con **al menos 2 imágenes/ventanas estacionales** (enero-marzo y
junio-agosto, las dos temporadas relativamente secas del régimen bimodal andino), para tener un
compuesto anual más robusto y con mejor cobertura libre de nubes que con una sola ventana.

## Estructura del notebook (una fase por cada objetivo específico)

| Fase | Objetivo específico | Contenido |
|---|---|---|
| **Fase 0** | — | Configuración del entorno (librerías y conexión a Google Earth Engine) |
| **Fase 1** | Recopilar imágenes satelitales multitemporales | AOI, selección de sensor por año, recopilación de 34 ventanas estacionales (17 años x 2) |
| **Fase 2** | Preprocesar las imágenes | Enmascarado de nubes específico por sensor, compuestos anuales robustos (S1+S2) e índices espectrales |
| **Fase 3** | Sistematizar la detección de cambios | Funciones reutilizables de diferencia/umbral/clasificación, comparación principal y año a año (16 pares) |
| **Fase 4** | Evaluar y validar los resultados | Estadísticas de área, validación cruzada multiclase (matriz de confusión, OA, AA, Kappa) y análisis de sensibilidad |
| **Fase 5** | Generar visualizaciones | Mapa interactivo, comparaciones antes/después, panel año a año, series de tiempo y exportación |

Cada fase explica, para sus pasos principales: **qué** se hizo, **cómo**, **por qué**, **qué
herramientas** se usaron y **qué resultado arroja** — pensado para sustentar un trabajo de
investigación académica (tesis de grado).

## Antes de empezar

1. **No necesitas GPU.** Todo el procesamiento pesado ocurre en los servidores de Google Earth Engine; Colab solo orquesta las solicitudes.
2. Necesitas una **cuenta de Google Earth Engine** (gratuita para uso no comercial/académico):
   - Regístrate en https://code.earthengine.google.com/register si no lo has hecho antes.
   - Crea (o reutiliza) un **proyecto de Google Cloud** asociado a Earth Engine; necesitarás su *ID de proyecto* en la Fase 0.
3. Ejecuta las celdas **en orden, de arriba hacia abajo** (▶️). Cada fase depende de las variables creadas en las fases anteriores.
4. El área de estudio, los años y las ventanas estacionales son **parámetros configurables** (Fase 1).
5. **Tiempo de ejecución:** con 17 años x 2 ventanas estacionales, más de un centenar de llamadas a
   Earth Engine se ejecutan en total a lo largo del notebook (recopilación, umbrales, estadísticas,
   miniaturas). Una ejecución completa puede tardar **entre 20 y 40 minutos**. Es el costo esperado
   de un análisis multitemporal denso de 17 años — normal para una tesis, no un error. Si necesitas
   iterar más rápido mientras pruebas el notebook, reduce temporalmente `AÑOS_ANALISIS` (Fase 1.3) y
   luego vuelve a ampliarlo para la corrida final.
6. La Fase 1.7 guarda automáticamente el conjunto de imágenes recopiladas (2 por año) en tu Google
   Drive, en `MyDrive/deteccion_cambios_bucaramanga/01_imagenes_recopiladas/` — esa carpeta ya
   existe y coincide con `CARPETA_RESULTADOS` (Fase 0.3).
7. **Limitación metodológica a tener en cuenta:** la comparación año a año que cruza 2016→2017 mezcla
   dos sensores distintos (Landsat 30 m → Sentinel-2 10 m). Se señala explícitamente en la Fase 3.6:
   interpreta ese intervalo con más cautela que el resto, donde el sensor no cambia.

## Fase 0. Configuración del entorno

**Qué se hizo:** instalar las librerías necesarias y autenticar la sesión contra Google Earth
Engine (GEE).
**Por qué:** GEE es la plataforma de acceso abierto que aloja simultáneamente los catálogos de
Landsat y Sentinel-2 ya calibrados, y ejecuta en la nube todo el álgebra de bandas pesada — evita
tener que descargar manualmente decenas de escenas satelitales.
**Herramientas:** `earthengine-api` (cliente Python oficial de GEE), `geemap` (mapas interactivos
sobre GEE en Colab/Jupyter).

### 0.1 Instalar y cargar librerías

In [ ]:
!pip install -q -U earthengine-api geemap

import ee
import geemap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import requests
from io import BytesIO
from PIL import Image
import os

print("Librerías cargadas correctamente.")

### 0.2 Autenticar y conectar con Google Earth Engine

Al ejecutar la siguiente celda se abrirá una ventana/enlace para iniciar sesión con tu cuenta de
Google y autorizar el acceso. Luego, reemplaza `EE_PROJECT_ID` por el **ID de tu proyecto de Google
Cloud** vinculado a Earth Engine (lo encuentras en https://code.earthengine.google.com/, arriba a la
izquierda, o en https://console.cloud.google.com/).

In [ ]:
EE_PROJECT_ID = "tu-proyecto-gee"  # <-- reemplaza con el ID de tu proyecto de Google Cloud / Earth Engine

ee.Authenticate()
ee.Initialize(project=EE_PROJECT_ID)

print("Conexión con Google Earth Engine establecida.")
print(f"Proyecto activo: {EE_PROJECT_ID}")

### 0.3 Carpeta de resultados (Google Drive)

Montamos Google Drive para guardar de forma persistente las imágenes recopiladas (Fase 1), las
figuras y las tablas de estadísticas (Fases 4 y 5).

In [ ]:
from google.colab import drive

MONTAR_DRIVE = True  # pon False si no quieres usar Drive (los resultados solo quedarán en la sesión de Colab)

if MONTAR_DRIVE:
    drive.mount("/content/drive")
    CARPETA_RESULTADOS = "/content/drive/MyDrive/deteccion_cambios_bucaramanga"
else:
    CARPETA_RESULTADOS = "/content/deteccion_cambios_bucaramanga"

os.makedirs(CARPETA_RESULTADOS, exist_ok=True)
print(f"Los resultados se guardarán en: {CARPETA_RESULTADOS}")

## Fase 1. Recopilación de imágenes satelitales multitemporales

**Objetivo específico:** *Recopilar un conjunto de imágenes satelitales multitemporales de una zona
geográfica específica, mediante el uso de plataformas de acceso abierto, como fuente de análisis de
los cambios en el terreno.*

**Qué se hizo:** definir el área de estudio (AMB), elegir automáticamente la plataforma satelital
correcta para cada año del rango 2010-2026, y recopilar al menos 2 imágenes (ventanas estacionales)
por año para esa plataforma.
**Cómo:** consultando los catálogos `LANDSAT/.../C02/T1_L2` y `COPERNICUS/S2_SR_HARMONIZED` de
Google Earth Engine, filtrados por el AOI, por fecha y por porcentaje de nubosidad de la escena.
**Por qué:** ninguna plataforma de acceso abierto por sí sola cubre con buena resolución los 17 años
solicitados; hay que combinar varias fuentes de forma controlada y documentada.
**Herramientas:** Google Earth Engine (catálogos Landsat Collection 2 Level 2 y Sentinel-2 SR
Harmonized), `earthengine-api`, `geemap`.
**Qué arroja:** un diccionario `COLECCIONES` con 34 colecciones de imágenes (una por año x ventana
estacional), listas para preprocesar en la Fase 2.

### 1.1 Definir el área de estudio (AMB)

In [ ]:
# Rectángulo delimitador del Área Metropolitana de Bucaramanga (AMB): cubre el casco urbano
# y la franja de expansión periurbana de los 4 municipios (Bucaramanga, Floridablanca, Girón
# y Piedecuesta). Es el AOI (Area Of Interest) principal usado en todo el notebook.
AMB_LON_MIN, AMB_LON_MAX = -73.22, -73.00
AMB_LAT_MIN, AMB_LAT_MAX = 6.93, 7.20

aoi_bbox = ee.Geometry.Rectangle(
    [AMB_LON_MIN, AMB_LAT_MIN, AMB_LON_MAX, AMB_LAT_MAX]
)

# Centro aproximado del AMB (Bucaramanga), usado para centrar los mapas interactivos.
AMB_CENTRO = [7.1193, -73.1227]

area_km2 = aoi_bbox.area().divide(1e6).getInfo()
print(f"Área de estudio (rectángulo delimitador del AMB): {area_km2:,.1f} km²")

### 1.2 (Opcional) Refinar el área con límites administrativos oficiales

Esta celda intenta reemplazar el rectángulo anterior por la **unión de los límites municipales
oficiales** (Bucaramanga, Floridablanca, Girón y Piedecuesta) tomados del conjunto de datos
`FAO/GAUL/2015/level2`. Si el conjunto de datos no está disponible, los nombres no coinciden, o el
resultado no pasa una **validación de tamaño** (para blindarnos contra una coincidencia de nombre
incorrecta que traiga un polígono lejano y agrande muchísimo el área), el notebook **conserva
automáticamente** el rectángulo delimitador de la celda anterior (`aoi_bbox`) — esta celda es
segura de ejecutar y no rompe el resto del flujo.

También se define `aoi_bounds`, el rectángulo delimitador **de la geometría final** (`aoi.bounds()`).
Se usa únicamente para descargar miniaturas/imágenes de vista previa: así se evita que, al recortar
con un polígono irregular, la miniatura quede con grandes zonas en blanco fuera del polígono pero
dentro de su rectángulo — el análisis (Fase 2 en adelante) sigue usando el polígono preciso `aoi`.

In [ ]:
MUNICIPIOS_AMB = ["Bucaramanga", "Floridablanca", "Giron", "Girón", "Piedecuesta"]

aoi = aoi_bbox  # valor por defecto: se sobrescribe abajo solo si el refinamiento funciona y pasa la validación
aoi_fuente = "Rectángulo delimitador (definido manualmente)"

# Rango de área aceptado para la geometría refinada, tomando como referencia el rectángulo de 1.1.
# Si el resultado de FAO GAUL cae fuera de este rango, es señal de una coincidencia de nombre
# incorrecta (p. ej. un municipio homónimo en otro lugar) y se descarta por seguridad.
FACTOR_AREA_MIN, FACTOR_AREA_MAX = 0.2, 4.0

try:
    limites_municipales = (
        ee.FeatureCollection("FAO/GAUL/2015/level2")
        .filter(ee.Filter.eq("ADM0_NAME", "Colombia"))
        .filter(ee.Filter.eq("ADM1_NAME", "Santander"))
        .filter(ee.Filter.inList("ADM2_NAME", MUNICIPIOS_AMB))
    )
    n_municipios = limites_municipales.size().getInfo()
    if n_municipios > 0:
        aoi_admin = limites_municipales.geometry().dissolve()
        area_admin_km2 = aoi_admin.area().divide(1e6).getInfo()
        razon_area = area_admin_km2 / area_km2

        if FACTOR_AREA_MIN <= razon_area <= FACTOR_AREA_MAX:
            aoi = aoi_admin
            aoi_fuente = f"Límites administrativos oficiales (FAO GAUL, {n_municipios} municipios encontrados)"
            print(f"AOI refinada con límites oficiales: {area_admin_km2:,.1f} km² (válida)")
        else:
            print(
                f"El área de FAO GAUL ({area_admin_km2:,.1f} km²) es sospechosamente distinta al "
                f"rectángulo delimitador ({area_km2:,.1f} km²) — probable coincidencia de nombre "
                "incorrecta. Se conserva el rectángulo delimitador."
            )
    else:
        print("No se encontraron los municipios por nombre en FAO GAUL; se conserva el rectángulo delimitador.")
except Exception as error:
    print(f"No fue posible refinar el AOI ({error}). Se conserva el rectángulo delimitador.")

aoi_bounds = aoi.bounds()

print(f"AOI activa (análisis): {aoi_fuente}")

### 1.3 Definir los años y las ventanas estacionales de análisis

**Qué se hizo:** definir el rango de años (2010-2026) y, para cada año, dos ventanas estacionales
(enero-marzo y junio-agosto).
**Por qué dos ventanas por año:** Bucaramanga tiene un régimen de lluvias **bimodal** (dos
temporadas relativamente secas: dic-mar y jun-ago). Usar únicamente una ventana deja al compuesto
anual a merced de la nubosidad de esos tres meses; con dos ventanas independientes, la Fase 2
dispone de más escenas para construir una mediana libre de nubes más confiable — y de paso se
cumple el requisito de tener **al menos 2 imágenes por año**.
**Por qué el rango 2010-2026:** es el rango de estudio solicitado; determina, a su vez, qué
plataforma satelital corresponde a cada año (Fase 1.4).

In [ ]:
AÑOS_ANALISIS = list(range(2010, 2027))  # 2010, 2011, ..., 2026 -> 17 años

VENTANAS_ESTACIONALES = {
    "S1": ("01-01", "03-31"),  # primera temporada relativamente seca (régimen bimodal andino)
    "S2": ("06-01", "08-31"),  # segunda temporada relativamente seca
}

# Diccionario {etiqueta: (fecha_inicio, fecha_fin)} con una entrada por año x ventana,
# p. ej. "2010-S1" -> ("2010-01-01", "2010-03-31"). 17 años x 2 ventanas = 34 periodos.
PERIODOS = {}
for anio in AÑOS_ANALISIS:
    for ventana, (mes_dia_inicio, mes_dia_fin) in VENTANAS_ESTACIONALES.items():
        PERIODOS[f"{anio}-{ventana}"] = (f"{anio}-{mes_dia_inicio}", f"{anio}-{mes_dia_fin}")

# Porcentaje máximo de nubosidad permitido por escena (metadato de la colección). Se usa un valor
# más permisivo que en un análisis de un solo sensor porque Landsat, en años/ventanas puntuales
# (p. ej. 2012, Landsat 7 con el defecto SLC-off), tiene menos escenas disponibles para elegir.
MAX_NUBES_ESCENA = 60

ETIQUETAS_PERIODOS = list(PERIODOS.keys())           # 34 ventanas estacionales (recopilación, Fase 1)
ETIQUETAS_ANIOS = [str(a) for a in AÑOS_ANALISIS]    # 17 años (nivel de comparación de cambios, Fase 3+)
T_INICIAL_LABEL = ETIQUETAS_ANIOS[0]
T_FINAL_LABEL = ETIQUETAS_ANIOS[-1]

# Pares de años consecutivos, p. ej. [("2010", "2011"), ..., ("2025", "2026")], usados en las
# Fases 3.6 / 4.2 / 5.3 para la comparación año a año.
PARES_CONSECUTIVOS = list(zip(ETIQUETAS_ANIOS[:-1], ETIQUETAS_ANIOS[1:]))

print(f"Ventanas estacionales definidas: {len(ETIQUETAS_PERIODOS)} ({len(ETIQUETAS_ANIOS)} años x {len(VENTANAS_ESTACIONALES)} ventanas/año)")
print(f"Comparación principal (todo el rango): {T_INICIAL_LABEL} → {T_FINAL_LABEL}")
print(f"Comparaciones año a año: {len(PARES_CONSECUTIVOS)}")

### 1.4 Selección automática de sensor por año y armonización de bandas

**Qué se hizo:** una función `config_sensor(anio)` que, para cualquier año del rango, devuelve la
colección de Earth Engine correcta, el nombre de la propiedad de nubosidad de esa colección, y un
**mapeo de bandas nativas a nombres comunes** (`BLUE`, `GREEN`, `RED`, `NIR`, `SWIR1`).

**Por qué la armonización de bandas:** Landsat y Sentinel-2 numeran sus bandas de forma distinta
para la misma longitud de onda (p. ej. el rojo es `SR_B3` en Landsat 5/7, `SR_B4` en Landsat 8/9, y
`B4` en Sentinel-2). Renombrarlas una sola vez aquí, a la entrada del pipeline, permite que **todo
el resto del notebook** (índices espectrales, diferencias, umbrales, clasificación) sea agnóstico
al sensor: la misma fórmula `NDVI = (NIR − RED) / (NIR + RED)` funciona sin importar de qué año
provienen las bandas.

| Sensor | Colección GEE | Bandas nativas usadas | Nube (metadato) |
|---|---|---|---|
| Landsat 5 TM | `LANDSAT/LT05/C02/T1_L2` | SR_B1,B2,B3,B4,B5 + QA_PIXEL | `CLOUD_COVER` |
| Landsat 7 ETM+ | `LANDSAT/LE07/C02/T1_L2` | SR_B1,B2,B3,B4,B5 + QA_PIXEL | `CLOUD_COVER` |
| Landsat 8 OLI | `LANDSAT/LC08/C02/T1_L2` | SR_B2,B3,B4,B5,B6 + QA_PIXEL | `CLOUD_COVER` |
| Sentinel-2 SR | `COPERNICUS/S2_SR_HARMONIZED` | B2,B3,B4,B8,B11 + SCL | `CLOUDY_PIXEL_PERCENTAGE` |

**Herramientas:** `ee.ImageCollection.select(bandas_nativas, bandas_comunes)` de Earth Engine, que
renombra bandas en una sola llamada sin descargar ni mover datos (toda la operación queda diferida
en el servidor de GEE).
**Qué arroja:** la función `config_sensor` y el diccionario `CONFIG_POR_ANIO`, usados en el resto
de la Fase 1 y en la Fase 2 para saber qué lógica de enmascarado de nubes aplicar a cada año.

In [ ]:
def config_sensor(anio):
    """Configuración de la plataforma satelital a usar para un año dado del estudio: colección de
    Earth Engine, mapeo de bandas nativas -> comunes (BLUE/GREEN/RED/NIR/SWIR1 + banda de calidad),
    nombre de la propiedad de nubosidad, resolución nativa y un rango de estiramiento (min/max)
    razonable para una vista previa en color verdadero sin procesar."""
    if anio <= 2011:
        return {
            "nombre": "Landsat 5 TM", "familia": "landsat", "resolucion_m": 30,
            "coleccion": "LANDSAT/LT05/C02/T1_L2",
            "mapa_bandas": {"SR_B1": "BLUE", "SR_B2": "GREEN", "SR_B3": "RED", "SR_B4": "NIR", "SR_B5": "SWIR1", "QA_PIXEL": "QA_PIXEL"},
            "prop_nubes": "CLOUD_COVER",
            "vis_crudo": {"min": 7000, "max": 20000},
        }
    elif anio == 2012:
        return {
            "nombre": "Landsat 7 ETM+ (SLC-off)", "familia": "landsat", "resolucion_m": 30,
            "coleccion": "LANDSAT/LE07/C02/T1_L2",
            "mapa_bandas": {"SR_B1": "BLUE", "SR_B2": "GREEN", "SR_B3": "RED", "SR_B4": "NIR", "SR_B5": "SWIR1", "QA_PIXEL": "QA_PIXEL"},
            "prop_nubes": "CLOUD_COVER",
            "vis_crudo": {"min": 7000, "max": 20000},
        }
    elif anio <= 2016:
        return {
            "nombre": "Landsat 8 OLI", "familia": "landsat", "resolucion_m": 30,
            "coleccion": "LANDSAT/LC08/C02/T1_L2",
            "mapa_bandas": {"SR_B2": "BLUE", "SR_B3": "GREEN", "SR_B4": "RED", "SR_B5": "NIR", "SR_B6": "SWIR1", "QA_PIXEL": "QA_PIXEL"},
            "prop_nubes": "CLOUD_COVER",
            "vis_crudo": {"min": 7000, "max": 20000},
        }
    else:
        return {
            "nombre": "Sentinel-2 SR Harmonized", "familia": "sentinel2", "resolucion_m": 10,
            "coleccion": "COPERNICUS/S2_SR_HARMONIZED",
            "mapa_bandas": {"B2": "BLUE", "B3": "GREEN", "B4": "RED", "B8": "NIR", "B11": "SWIR1", "SCL": "SCL"},
            "prop_nubes": "CLOUDY_PIXEL_PERCENTAGE",
            "vis_crudo": {"min": 0, "max": 3000},
        }


CONFIG_POR_ANIO = {anio: config_sensor(anio) for anio in AÑOS_ANALISIS}

resumen_sensores = pd.DataFrame([
    {"anio": anio, "sensor": cfg["nombre"], "resolucion_m": cfg["resolucion_m"]}
    for anio, cfg in CONFIG_POR_ANIO.items()
])
resumen_sensores

### 1.5 Recopilar las colecciones para cada año x ventana estacional

**Qué se hizo:** para cada una de las 34 combinaciones año-ventana, se consulta el catálogo de
Earth Engine que corresponde a ese año (Fase 1.4), filtrado por el AOI, por la fecha y por el
porcentaje máximo de nubosidad, y se renombran sus bandas a los nombres comunes.
**Cómo:** función `obtener_coleccion`, que aplica `filterBounds` + `filterDate` + un filtro de
nubosidad + `select(bandas_nativas, bandas_comunes)`.
**Por qué:** es el paso central del objetivo de la Fase 1 — "recopilar un conjunto de imágenes
satelitales multitemporales... mediante plataformas de acceso abierto".
**Qué arroja:** el diccionario `COLECCIONES` (34 entradas) y un conteo impreso de cuántas escenas
se encontraron en cada ventana — útil para detectar de inmediato años/ventanas con poca disponibilidad.

In [ ]:
def obtener_coleccion(anio, fecha_inicio, fecha_fin, aoi, max_nubes):
    """Filtra y renombra (a bandas comunes) la colección de Earth Engine que corresponde al año
    dado, según config_sensor. Devuelve (coleccion, cfg)."""
    cfg = config_sensor(anio)
    bandas_nativas = list(cfg["mapa_bandas"].keys())
    bandas_comunes = list(cfg["mapa_bandas"].values())

    coleccion = (
        ee.ImageCollection(cfg["coleccion"])
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .filter(ee.Filter.lt(cfg["prop_nubes"], max_nubes))
        .select(bandas_nativas, bandas_comunes)
    )
    return coleccion, cfg


COLECCIONES = {}

for anio in AÑOS_ANALISIS:
    for ventana in VENTANAS_ESTACIONALES:
        etiqueta = f"{anio}-{ventana}"
        fecha_inicio, fecha_fin = PERIODOS[etiqueta]
        coleccion, cfg = obtener_coleccion(anio, fecha_inicio, fecha_fin, aoi, MAX_NUBES_ESCENA)
        n_imagenes = coleccion.size().getInfo()
        COLECCIONES[etiqueta] = coleccion

        estado = "✅" if n_imagenes > 0 else "⚠️ sin imágenes: amplía la ventana o sube MAX_NUBES_ESCENA"
        print(f"{etiqueta} [{cfg['nombre']}]: {n_imagenes} imágenes {estado}")

n_vacias = sum(1 for etq in COLECCIONES if COLECCIONES[etq].size().getInfo() == 0)
print(f"\n{len(COLECCIONES)} ventanas recopiladas ({n_vacias} sin imágenes).")

### 1.6 Vista previa rápida (color verdadero) de una ventana

Igual que antes: una sola escena rara vez cubre todo el AOI (borde de la franja de barrido del
satélite), así que se usa un **mosaico** de todas las escenas de la ventana. `obtener_mosaico_periodo`
también devuelve la configuración del sensor (`cfg`), necesaria para usar el rango de estiramiento
de color correcto — Landsat y Sentinel-2 codifican la reflectancia sin procesar en escalas distintas.

**Nota sobre el mapa base:** todos los mapas interactivos del notebook usan `basemap="ROADMAP"`
(las teselas de Google Maps que `geemap` trae integradas) en vez del mapa base de OpenStreetMap
que trae por defecto. Los servidores voluntarios de OpenStreetMap bloquean con frecuencia el acceso
automatizado desde Colab ("Access blocked"), y proveedores alternativos como CartoDB movieron sus
estilos gratuitos detrás de una API key en 2023-2024 ("API KEY REQUIRED" en las teselas). El
basemap de Google no requiere ninguna clave y es el que usa la mayoría de los tutoriales oficiales
de `geemap`. Esto solo afecta el fondo del mapa — las capas propias (imágenes satelitales, NDVI,
mapa de cambios) se siguen renderizando igual, vengan de Earth Engine. Si por algún motivo tampoco
carga, prueba `basemap="HYBRID"` o `basemap="TERRAIN"` (mismo proveedor, otro estilo).

In [ ]:
def obtener_mosaico_periodo(etiqueta):
    """Mosaico de todas las escenas de una ventana año-estación: la escena menos nubosa queda
    visible arriba, y el resto rellena lo que esa escena no cubre. Devuelve (mosaico, cfg)."""
    anio = int(etiqueta.split("-")[0])
    cfg = config_sensor(anio)
    mosaico = COLECCIONES[etiqueta].sort(cfg["prop_nubes"], False).mosaic()
    return mosaico, cfg

etiqueta_preview = f"{T_FINAL_LABEL}-S1"
imagen_preliminar, cfg_preview = obtener_mosaico_periodo(etiqueta_preview)
vis_rgb_cruda = {"bands": ["RED", "GREEN", "BLUE"], **cfg_preview["vis_crudo"]}

mapa_previo = geemap.Map(center=AMB_CENTRO, zoom=11, basemap="ROADMAP")
mapa_previo.addLayer(imagen_preliminar, vis_rgb_cruda, f"{cfg_preview['nombre']} sin procesar ({etiqueta_preview})")
mapa_previo.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["red"]}, "Límite del AOI (AMB)")
mapa_previo

### 1.7 Guardar en Google Drive el conjunto de imágenes recopiladas

**Qué se hizo:** descargar una imagen en color verdadero por cada una de las 34 ventanas (2 por
año) y guardarla como PNG en Google Drive.
**Por qué:** para poder **ver directamente el conjunto de imágenes multitemporales** recopilado,
sin depender del mapa interactivo, y como evidencia documental del insumo de entrada del análisis.
**Herramientas:** `Image.getThumbURL` de Earth Engine + `requests` para la descarga + `Pillow`
(PIL) para guardar el PNG. La función `obtener_imagen_ee` se reutiliza más adelante en la Fase 5.
**Qué arroja:** 34 archivos PNG en
`MyDrive/deteccion_cambios_bucaramanga/01_imagenes_recopiladas/`, dos por año (`S1` y `S2`),
nombrados con el año, la ventana y el sensor usado.

In [ ]:
def obtener_imagen_ee(imagen, vis_params, dimensiones=1024):
    """Descarga una imagen de Earth Engine como objeto PIL.Image, lista para guardar en disco o
    convertir a arreglo numpy para graficar. Usa `aoi_bounds` (rectángulo) en vez de `aoi`
    (que puede ser un polígono irregular) para que la miniatura salga completa, sin zonas en
    blanco fuera del polígono pero dentro de su rectángulo delimitador."""
    url = imagen.getThumbURL({**vis_params, "region": aoi_bounds, "dimensions": dimensiones, "format": "png"})
    respuesta = requests.get(url, timeout=60)
    respuesta.raise_for_status()
    return Image.open(BytesIO(respuesta.content))


CARPETA_IMAGENES_RECOPILADAS = os.path.join(CARPETA_RESULTADOS, "01_imagenes_recopiladas")
os.makedirs(CARPETA_IMAGENES_RECOPILADAS, exist_ok=True)

for etiqueta in ETIQUETAS_PERIODOS:
    mosaico, cfg = obtener_mosaico_periodo(etiqueta)
    n_escenas = COLECCIONES[etiqueta].size().getInfo()
    vis = {"bands": ["RED", "GREEN", "BLUE"], **cfg["vis_crudo"]}

    imagen_png = obtener_imagen_ee(mosaico, vis)
    sensor_corto = cfg["nombre"].split()[0].replace("(", "")
    ruta_imagen = os.path.join(CARPETA_IMAGENES_RECOPILADAS, f"{etiqueta}_{sensor_corto}.png")
    imagen_png.save(ruta_imagen)
    print(f"{etiqueta} [{cfg['nombre']}]: guardada {os.path.basename(ruta_imagen)} (mosaico de {n_escenas} escenas)")

print(f"\nConjunto de imágenes multitemporales guardado en Google Drive: {CARPETA_IMAGENES_RECOPILADAS}")

## Fase 2. Preprocesamiento de las imágenes satelitales

**Objetivo específico:** *Preprocesar las imágenes satelitales recopiladas, aplicando la técnica de
análisis multitemporal en la detección de cambios en el terreno, como expansión urbana,
deforestación o impacto de desastres naturales.*

**Qué se hizo:** (2.1) enmascarar nubes y sombras con la lógica propia de cada sensor; (2.2)
combinar las dos ventanas estacionales de cada año en **un único compuesto anual** mediante la
mediana; (2.3) calcular los índices espectrales NDVI, NDBI y NDWI sobre cada compuesto anual.
**Cómo:** todo el álgebra de bandas se ejecuta en el servidor de Earth Engine (`ee.Image`,
`ee.ImageCollection.map/median`); Python solo orquesta qué imágenes combinar.
**Por qué:** las escenas crudas de satélite traen nubes, sombras y ruido atmosférico residual que
invalidarían una comparación pixel a pixel; el compuesto mediano de varias escenas suprime esos
artefactos, y los índices espectrales resumen la señal óptica de 5 bandas en 3 variables
interpretables (vegetación, suelo construido, agua) — la base para detectar cambios en la Fase 3.
**Herramientas:** Google Earth Engine (`updateMask`, `bitwiseAnd`, `normalizedDifference`,
`ImageCollection.merge/median`).
**Qué arroja:** el diccionario `COMPOSITES`, con 17 compuestos anuales (2010-2026), cada uno con
5 bandas de reflectancia armonizadas (`BLUE/GREEN/RED/NIR/SWIR1`) + 3 bandas de índices
(`NDVI/NDBI/NDWI`), listos para la Fase 3.

### 2.1 Enmascarado de nubes y sombras (específico por sensor)

**Qué se hizo:** dos funciones de enmascarado, una por familia de sensor, que además escalan la
reflectancia a su rango físico [0, 1]:

- **Sentinel-2** (`enmascarar_sentinel2`): usa la banda `SCL` (Scene Classification Layer) para
  excluir sombra de nube, nube de probabilidad media/alta y cirro delgado; escala con el factor
  oficial `DN / 10000`.
- **Landsat 5/7/8** (`enmascarar_landsat`): usa los **bits de calidad de `QA_PIXEL`** (Collection 2
  Level 2) para excluir nube diluida, cirro, nube y sombra de nube; escala con el factor oficial de
  USGS `DN × 0.0000275 − 0.2`.

**Por qué dos funciones y no una sola:** cada agencia espacial codifica la calidad del píxel de
forma distinta (SCL es una banda categórica de ESA; QA_PIXEL es un entero de bits de USGS) — no son
intercambiables, así que se respeta la metodología oficial de cada sensor en vez de forzar un único
criterio genérico que perdería precisión.
**Qué arroja:** dos funciones (`enmascarar_sentinel2`, `enmascarar_landsat`) que, aplicadas a una
imagen cruda de su familia correspondiente, devuelven las 5 bandas de reflectancia ya enmascaradas.

In [ ]:
def enmascarar_sentinel2(imagen):
    """Enmascara nubes, sombras de nubes y cirros usando la banda SCL (Scene Classification) de
    Sentinel-2, y escala las bandas ópticas a reflectancia de superficie [0, 1] (factor oficial
    DN / 10000)."""
    scl = imagen.select("SCL")
    # Clases SCL a excluir: 3 sombra de nube, 8 nube prob. media, 9 nube prob. alta, 10 cirro delgado.
    mascara_valida = scl.neq(3).And(scl.neq(8)).And(scl.neq(9)).And(scl.neq(10))

    bandas_opticas = imagen.select(["BLUE", "GREEN", "RED", "NIR", "SWIR1"]).multiply(0.0001)
    return bandas_opticas.updateMask(mascara_valida).copyProperties(imagen, imagen.propertyNames())


def enmascarar_landsat(imagen):
    """Enmascara nubes, sombras de nubes y cirros usando los bits de calidad de QA_PIXEL
    (Landsat Collection 2 Level 2), y escala las bandas ópticas a reflectancia de superficie
    [0, 1] con el factor oficial de USGS: DN x 0.0000275 - 0.2."""
    qa = imagen.select("QA_PIXEL")
    mascara_valida = ee.Image(1)
    for bit in [1, 2, 3, 4]:  # 1 nube diluida, 2 cirro, 3 nube, 4 sombra de nube
        mascara_valida = mascara_valida.And(qa.bitwiseAnd(1 << bit).eq(0))

    bandas_opticas = imagen.select(["BLUE", "GREEN", "RED", "NIR", "SWIR1"]).multiply(0.0000275).add(-0.2)
    return bandas_opticas.updateMask(mascara_valida).copyProperties(imagen, imagen.propertyNames())


def funcion_mascara_para(cfg):
    """Elige la función de enmascarado correcta según la familia de sensor del año (ver Fase 1.4)."""
    return enmascarar_sentinel2 if cfg["familia"] == "sentinel2" else enmascarar_landsat

print("Funciones de enmascarado de nubes definidas (Sentinel-2 y Landsat).")

### 2.2 Generar el compuesto anual robusto (S1 + S2, mediana)

**Qué se hizo:** para cada año, se combinan (`merge`) las colecciones de las dos ventanas
estacionales (S1 y S2), se enmascaran nubes en cada escena con la función que corresponde a su
sensor, y se calcula la **mediana** del conjunto combinado.
**Por qué combinar S1+S2 en vez de usar una sola ventana:** más escenas de entrada producen una
mediana más robusta frente a nubes residuales y outliers — especialmente importante en años con
pocas escenas disponibles, como 2012 (Landsat 7 con el defecto *Scan Line Corrector* apagado, que
deja franjas sin datos en cada escena individual).
**Herramientas:** `ee.ImageCollection.merge`, `.map()` (aplica el enmascarado a cada escena),
`.median()` (compuesto por pixel), `.clip(aoi)` (recorte al área de estudio).
**Qué arroja:** el diccionario `COMPOSITES` (17 compuestos anuales, 2010-2026) y, por cada año, el
número total de escenas (S1+S2) que entraron al compuesto — una medida directa de cuántos datos
respaldan cada año del análisis.

In [ ]:
COMPOSITES = {}

for anio in AÑOS_ANALISIS:
    etiqueta_anio = str(anio)
    cfg = CONFIG_POR_ANIO[anio]
    funcion_mascara = funcion_mascara_para(cfg)

    coleccion_anual = COLECCIONES[f"{etiqueta_anio}-S1"].merge(COLECCIONES[f"{etiqueta_anio}-S2"])
    n_total = coleccion_anual.size().getInfo()

    composite = coleccion_anual.map(funcion_mascara).median().clip(aoi)
    COMPOSITES[etiqueta_anio] = composite

    print(f"{etiqueta_anio} [{cfg['nombre']}]: compuesto anual a partir de {n_total} escenas (S1+S2)")

print(f"\nCompuestos anuales generados: {list(COMPOSITES.keys())}")

### 2.3 Cálculo de índices espectrales

Con las bandas ya armonizadas (Fase 1.4), la misma fórmula se aplica sin importar el sensor de
origen del compuesto:

- **NDVI** = (NIR − RED) / (NIR + RED) → vigor y cobertura de vegetación.
- **NDBI** = (SWIR1 − NIR) / (SWIR1 + NIR) → superficies construidas / suelo urbano.
- **NDWI** = (GREEN − NIR) / (GREEN + NIR) → cuerpos de agua (McFeeters, 1996).

**Herramientas:** `ee.Image.normalizedDifference`, que calcula (A−B)/(A+B) en una sola llamada.
**Qué arroja:** los mismos 17 compuestos anuales, ahora con 3 bandas adicionales (`NDVI`, `NDBI`,
`NDWI`) — la entrada directa de la Fase 3.

In [ ]:
def calcular_indices(imagen):
    """Agrega al compuesto las bandas de índices espectrales NDVI, NDBI y NDWI, calculadas con
    las bandas comunes armonizadas (Fase 1.4), sin importar el sensor de origen."""
    ndvi = imagen.normalizedDifference(["NIR", "RED"]).rename("NDVI")
    ndbi = imagen.normalizedDifference(["SWIR1", "NIR"]).rename("NDBI")
    ndwi = imagen.normalizedDifference(["GREEN", "NIR"]).rename("NDWI")
    return imagen.addBands([ndvi, ndbi, ndwi])

COMPOSITES = {etiqueta: calcular_indices(img) for etiqueta, img in COMPOSITES.items()}

print("Índices NDVI, NDBI y NDWI calculados para cada compuesto anual.")
print("Bandas disponibles en cada compuesto:", COMPOSITES[T_FINAL_LABEL].bandNames().getInfo())

### 2.4 Verificación visual del preprocesamiento

Comparación rápida, sobre el mapa interactivo, entre el primer y el último año del estudio: color
verdadero y NDVI. Sirve como control de calidad antes de pasar a la Fase 3 — si el compuesto se ve
con vetas de nubes o franjas sin datos, conviene revisar `MAX_NUBES_ESCENA` (Fase 1.3) o las
ventanas estacionales.

In [ ]:
vis_rgb = {"bands": ["RED", "GREEN", "BLUE"], "min": 0, "max": 0.3}
vis_ndvi = {"bands": ["NDVI"], "min": -0.2, "max": 0.8, "palette": ["#a50026", "#ffffbf", "#1a9850"]}

mapa_preprocesado = geemap.Map(center=AMB_CENTRO, zoom=11, basemap="ROADMAP")
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_rgb, f"Color verdadero {T_INICIAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_rgb, f"Color verdadero {T_FINAL_LABEL} (sin nubes)")
mapa_preprocesado.addLayer(COMPOSITES[T_INICIAL_LABEL], vis_ndvi, f"NDVI {T_INICIAL_LABEL}", shown=False)
mapa_preprocesado.addLayer(COMPOSITES[T_FINAL_LABEL], vis_ndvi, f"NDVI {T_FINAL_LABEL}", shown=False)
mapa_preprocesado.addLayerControl()
mapa_preprocesado

## Fase 3. Sistematización de la detección de cambios

**Objetivo específico:** *Sistematizar la técnica de detección de los cambios en la cobertura
terrestre, identificando las variaciones significativas en el terreno especificado.*

**Qué se hizo:** empaquetar la metodología de detección de cambios en **tres funciones
reutilizables** (`calcular_deltas`, `obtener_umbrales`, `clasificar_cambios`), aplicarlas primero a
la **comparación principal** (2010 → 2026, todo el rango) y después, con el mismo código, a **cada
uno de los 16 pares de años consecutivos** (Fase 3.6).
**Cómo:** (1) diferencia de índices entre dos compuestos anuales; (2) umbral estadístico
*media ± k·desviación estándar*, calculado sobre el propio AOI; (3) combinación de los tres
índices umbralizados en un único mapa de 5 categorías.
**Por qué un método estadístico y no un umbral fijo elegido a ojo:** un umbral fijo (p. ej.
"ΔNDVI < −0.1") no se adapta a la variabilidad propia de cada par de años/sensores; el criterio
*media ± k·σ* se recalcula automáticamente para cada comparación, así que el **mismo criterio
objetivo** se aplica igual en 2010→2011 (Landsat) que en 2025→2026 (Sentinel-2) — esto es lo que
hace el método **sistemático y reproducible**, y no una simple inspección visual.
**Herramientas:** álgebra de bandas de Earth Engine (`subtract`, `abs`, `where`) y estadística
descriptiva vía `ee.Reducer.mean()/stdDev()`.
**Qué arroja:** `mapa_cambios` (mapa clasificado de la comparación principal) y
`MAPAS_CAMBIO_ANUAL` (16 mapas, uno por año consecutivo), todos con la misma escala de 5
categorías, listos para evaluarse en la Fase 4 y visualizarse en la Fase 5.

### 3.1 Compuestos del periodo inicial y final (comparación principal)

In [ ]:
imagen_inicial = COMPOSITES[T_INICIAL_LABEL]
imagen_final = COMPOSITES[T_FINAL_LABEL]

print(f"Periodo inicial: {T_INICIAL_LABEL} [{CONFIG_POR_ANIO[int(T_INICIAL_LABEL)]['nombre']}]")
print(f"Periodo final:   {T_FINAL_LABEL} [{CONFIG_POR_ANIO[int(T_FINAL_LABEL)]['nombre']}]")

### 3.2 Función de diferencia de índices espectrales (ΔNDVI, ΔNDBI, ΔNDWI)

Se define como función porque se reutiliza en la Fase 3.6 para cada comparación año a año, no solo
para la comparación principal.

In [ ]:
def calcular_deltas(imagen_ini, imagen_fin):
    """Diferencias de índices espectrales entre dos compuestos cualesquiera: ΔNDVI, ΔNDBI y ΔNDWI.
    Se reutiliza tanto para la comparación principal como para cada comparación año a año (3.6)."""
    delta_ndvi = imagen_fin.select("NDVI").subtract(imagen_ini.select("NDVI")).rename("delta_NDVI")
    delta_ndbi = imagen_fin.select("NDBI").subtract(imagen_ini.select("NDBI")).rename("delta_NDBI")
    delta_ndwi = imagen_fin.select("NDWI").subtract(imagen_ini.select("NDWI")).rename("delta_NDWI")
    return delta_ndvi.addBands([delta_ndbi, delta_ndwi])

deltas = calcular_deltas(imagen_inicial, imagen_final)
print("Bandas de diferencia calculadas (comparación principal):", deltas.bandNames().getInfo())

### 3.3 Umbral estadístico sistemático (media ± k·desviación estándar)

In [ ]:
K_UMBRAL = 1.5  # sensibilidad del umbral; se explora en la Fase 4.4 (análisis de sensibilidad)

def obtener_umbrales(imagen_delta, banda, aoi, escala=20, k=K_UMBRAL):
    """Calcula (umbral_inferior, umbral_superior) = media ± k·desv. estándar de una banda
    de diferencia, sobre el AOI. Es el mismo criterio estadístico para cualquier banda, zona o
    par de periodos: se reutiliza para la comparación principal y para cada año a año (3.6)."""
    estadisticas = imagen_delta.select(banda).reduceRegion(
        reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
        geometry=aoi,
        scale=escala,
        bestEffort=True,
        maxPixels=1e9,
    ).getInfo()

    media = estadisticas[f"{banda}_mean"]
    desviacion = estadisticas[f"{banda}_stdDev"]
    return media - k * desviacion, media + k * desviacion

umbral_ndvi_inf, umbral_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi)
umbral_ndbi_inf, umbral_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi)
umbral_ndwi_inf, umbral_ndwi_sup = obtener_umbrales(deltas, "delta_NDWI", aoi)

print(f"Umbral ΔNDVI: [{umbral_ndvi_inf:.4f}, {umbral_ndvi_sup:.4f}]  (pérdida de vegetación / ganancia de vegetación)")
print(f"Umbral ΔNDBI: [{umbral_ndbi_inf:.4f}, {umbral_ndbi_sup:.4f}]  (expansión urbana)")
print(f"Umbral ΔNDWI: [{umbral_ndwi_inf:.4f}, {umbral_ndwi_sup:.4f}]  (cambio hídrico)")

### 3.4 Análisis de Vector de Cambio (CVA) — magnitud del cambio

Complementa el análisis por índices individuales con una medida conjunta de "cuánto" cambió cada
píxel, combinando vegetación (NDVI) y superficie construida (NDBI) en un solo vector.

In [ ]:
magnitud_cambio = (
    deltas.select("delta_NDVI").pow(2)
    .add(deltas.select("delta_NDBI").pow(2))
    .sqrt()
    .rename("magnitud_CVA")
)

umbral_magnitud = magnitud_cambio.reduceRegion(
    reducer=ee.Reducer.mean().combine(ee.Reducer.stdDev(), sharedInputs=True),
    geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9,
).getInfo()

umbral_cva = umbral_magnitud["magnitud_CVA_mean"] + K_UMBRAL * umbral_magnitud["magnitud_CVA_stdDev"]
print(f"Umbral de magnitud de cambio (CVA): {umbral_cva:.4f} — píxeles por encima se consideran 'cambio relevante'")

### 3.5 Función de clasificación de cambios (y mapa de la comparación principal)

Se combina lo anterior en una **función reutilizable** (`clasificar_cambios`) que, dado cualquier
par de compuestos, calcula sus diferencias, obtiene los umbrales estadísticos y construye el mapa
de cambios clasificado en 5 categorías mutuamente excluyentes, aplicando los umbrales en orden de
prioridad (agua → vegetación → urbano → sin cambio):

| Código | Categoría | Criterio |
|---|---|---|
| 0 | Sin cambio significativo | ninguno de los criterios siguientes se cumple |
| 1 | Pérdida de vegetación (posible deforestación) | ΔNDVI < umbral inferior |
| 2 | Ganancia de vegetación | ΔNDVI > umbral superior |
| 3 | Expansión urbana / suelo construido | ΔNDBI > umbral superior |
| 4 | Cambio en cuerpos de agua | \|ΔNDWI\| > umbral superior |

Se aplica aquí a la **comparación principal** (todo el rango de estudio) y se reutiliza en la
Fase 3.6 para cada comparación año a año — garantizando que se aplique **exactamente el mismo
criterio metodológico** en ambos casos.

**Nota de visualización — que el cambio resalte:** en cualquier comparación de dos fechas, lo normal
(y esperable) es que la mayor parte del AOI caiga en la categoría 0 ("sin cambio"); si se pinta con
un color sólido, ese gris cubre casi todo el mapa y opaca visualmente las zonas donde sí hubo un
cambio real. Por eso, **solo para las visualizaciones** (nunca para el análisis: las estadísticas de
área, la validación y el mapa exportado siguen usando las 5 categorías completas, sin dilatar),
`preparar_para_mapa` hace dos cosas: (1) deja la categoría 0 transparente, para que el mapa base se
vea a través de las zonas sin cambio; y (2) dilata ligeramente los píxeles de cambio (`focal_max`)
para que un cluster pequeño no desaparezca al ver todo el AMB con zoom alejado. También se usa una
paleta de colores más saturada (rojo, verde, naranja y azul vivos) para maximizar el contraste
contra el mapa base, y se agrega la **imagen satelital real** del año final como capa de fondo
(en vez del mapa de calles genérico), para que el mapa se vea completo con el detalle real del
AMB — no solo el mapa base — y los cambios resalten con claridad encima.

In [ ]:
CATEGORIAS_CAMBIO = {
    0: ("Sin cambio significativo", "#d9d9d9"),
    1: ("Pérdida de vegetación (posible deforestación)", "#e31a1c"),
    2: ("Ganancia de vegetación", "#33a02c"),
    3: ("Expansión urbana / suelo construido", "#ff7f00"),
    4: ("Cambio en cuerpos de agua", "#1f78b4"),
}
paleta_cambios = [color for _, color in CATEGORIAS_CAMBIO.values()]


def preparar_para_mapa(mapa_clases, radio_resalte=1):
    """Prepara un mapa de clases SOLO para visualización (nunca para cálculos):
    1) enmascara la categoría 'sin cambio' (0), dejándola transparente para que el mapa base/la
       imagen de fondo se vea a través de ella y las categorías de cambio (1-4) resalten;
    2) dilata ligeramente los píxeles de cambio (`focal_max`) para que un cluster pequeño (a
       veces un solo píxel de 10-30 m) siga siendo visible al ver todo el AMB con zoom alejado,
       en vez de perderse en la pantalla. El mapa de clases original, sin dilatar y con la
       categoría 0 incluida, se sigue usando sin cambios para todo el análisis numérico
       (estadísticas de área, validación, mapa exportado)."""
    resaltado = mapa_clases.focal_max(radius=radio_resalte, kernelType="square") if radio_resalte else mapa_clases
    return resaltado.updateMask(resaltado.gt(0))


VIS_CAMBIOS = {"min": 1, "max": 4, "palette": paleta_cambios[1:]}  # sin la categoría 0 (transparente)


def clasificar_cambios(imagen_ini, imagen_fin, aoi, k=K_UMBRAL, escala=20):
    """Aplica la metodología sistematizada de la Fase 3 a cualquier par de compuestos: calcula
    las diferencias de índices, obtiene los umbrales estadísticos (media ± k·σ) y construye el
    mapa de cambios clasificado en 5 categorías. Devuelve (mapa_clasificado, umbrales) para poder
    inspeccionar los umbrales usados en cada comparación."""
    deltas_par = calcular_deltas(imagen_ini, imagen_fin)

    umb_ndvi = obtener_umbrales(deltas_par, "delta_NDVI", aoi, escala=escala, k=k)
    umb_ndbi = obtener_umbrales(deltas_par, "delta_NDBI", aoi, escala=escala, k=k)
    umb_ndwi = obtener_umbrales(deltas_par, "delta_NDWI", aoi, escala=escala, k=k)

    mapa = (
        ee.Image(0)
        .where(deltas_par.select("delta_NDVI").gt(umb_ndvi[1]), 2)   # ganancia de vegetación
        .where(deltas_par.select("delta_NDVI").lt(umb_ndvi[0]), 1)   # pérdida de vegetación
        .where(deltas_par.select("delta_NDBI").gt(umb_ndbi[1]), 3)   # expansión urbana
        .where(deltas_par.select("delta_NDWI").abs().gt(umb_ndwi[1]), 4)  # cambio hídrico
        .rename("clase_cambio")
        .clip(aoi)
        .updateMask(imagen_ini.select("NDVI").mask().And(imagen_fin.select("NDVI").mask()))
    )
    return mapa, {"NDVI": umb_ndvi, "NDBI": umb_ndbi, "NDWI": umb_ndwi}


mapa_cambios, umbrales_principal = clasificar_cambios(imagen_inicial, imagen_final, aoi)

mapa_clasificado_vis = geemap.Map(center=AMB_CENTRO, zoom=11, basemap="ROADMAP")
mapa_clasificado_vis.addLayer(
    imagen_final, vis_rgb, f"Color verdadero {T_FINAL_LABEL} (fondo)"
)  # imagen satelital real de fondo, para que el mapa no se vea vacío donde no hubo cambio
mapa_clasificado_vis.addLayer(
    preparar_para_mapa(mapa_cambios),
    VIS_CAMBIOS,
    f"Cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_clasificado_vis.add_legend(
    title="Categoría de cambio (0 se deja transparente)",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_clasificado_vis

### 3.6 Detección de cambios año a año (comparaciones consecutivas)

Para lograr una **comparación exitosa** entre lo ocurrido en cada año —y no solo entre el primero y
el último—, se aplica `clasificar_cambios` a **cada par de años consecutivos** definido en
`PARES_CONSECUTIVOS` (Fase 1.3): 2010→2011, 2011→2012, …, 2025→2026 (16 pares). El resultado es un
mapa de cambios independiente por cada intervalo de un año, calculado con **exactamente la misma
metodología** (mismos criterios, umbrales recalculados para cada par) que la comparación principal.

⚠️ **Advertencia metodológica:** el par **2016→2017** cruza el cambio de sensor documentado en la
Fase 1.4 (Landsat 30 m → Sentinel-2 10 m). Una parte de la diferencia detectada en ese intervalo
puede deberse al cambio de resolución/radiometría y no a un cambio real en el terreno — interprétalo
con más cautela que los demás pares, que sí comparan el mismo sensor.

In [ ]:
MAPAS_CAMBIO_ANUAL = {}
UMBRALES_ANUALES = {}

for anio_ini, anio_fin in PARES_CONSECUTIVOS:
    par_label = f"{anio_ini}→{anio_fin}"
    mapa_par, umbrales_par = clasificar_cambios(COMPOSITES[anio_ini], COMPOSITES[anio_fin], aoi)
    MAPAS_CAMBIO_ANUAL[par_label] = mapa_par
    UMBRALES_ANUALES[par_label] = umbrales_par

    aviso = "  ⚠️ cruza cambio de sensor (Landsat -> Sentinel-2)" if (anio_ini, anio_fin) == ("2016", "2017") else ""
    print(f"{par_label}: mapa de cambios calculado (umbral ΔNDBI = {umbrales_par['NDBI'][1]:.4f}){aviso}")

print(f"\n{len(MAPAS_CAMBIO_ANUAL)} mapas de cambio año a año calculados.")

## Fase 4. Evaluación y validación de los resultados

**Objetivo específico:** *Evaluar los resultados obtenidos, validando la confiabilidad de la
herramienta propuesta, mediante el análisis comparativo de las imágenes satelitales multitemporales.*

**Qué se hizo:** cuantificar (1) cuánta área cambió y en qué categoría, para la comparación
principal y año a año; (2) qué tan confiable es el mapa de cambios, comparándolo contra una fuente
de datos **independiente** (Dynamic World) con las métricas estándar de exactitud temática
(matriz de confusión, OA, AA, Kappa); y (3) qué tan sensible es el resultado a la elección del
parámetro `k` del umbral.
**Por qué:** un mapa de cambios sin una medida de confiabilidad es solo una hipótesis visual; esta
fase es la que le da **sustento y peso estadístico** al proyecto — permite afirmar, con una cifra
verificable, qué tan bien se desempeña la herramienta, en vez de solo mostrar mapas bonitos.
**Herramientas:** `ee.Reducer` (área por categoría), Google Dynamic World (`GOOGLE/DYNAMICWORLD/V1`,
producto de cobertura de suelo independiente, no usado en ningún paso anterior), `scikit-learn`
(`confusion_matrix`, `cohen_kappa_score`, `classification_report`) para las métricas de exactitud.
**Qué arroja:** tablas de área (`tabla_area`, `tabla_area_anual`), y las métricas de validación
**OA, AA y Kappa** con su matriz de confusión — el mismo esquema de resultados que se reporta en
benchmarks de clasificación de cobertura de suelo como **Indian Pines**.

### 4.1 Estadísticas de área por categoría de cambio (comparación principal)

In [ ]:
def calcular_area_por_categoria(mapa_clases, aoi, escala=20):
    """Área (ha) de cada categoría de un mapa de cambios clasificado, agregada sobre el AOI.
    Función reutilizable: se aplica aquí a la comparación principal y, en la Fase 4.2, a cada
    comparación año a año."""
    area_pixel = ee.Image.pixelArea().divide(10000)  # hectáreas por píxel
    resultado = (
        area_pixel.addBands(mapa_clases)
        .reduceRegion(
            reducer=ee.Reducer.sum().group(groupField=1, groupName="clase"),
            geometry=aoi, scale=escala, bestEffort=True, maxPixels=1e9,
        )
        .getInfo()
    )
    filas = []
    for grupo in resultado["groups"]:
        clase = int(grupo["clase"])
        nombre, color = CATEGORIAS_CAMBIO[clase]
        filas.append({"clase": clase, "categoria": nombre, "hectareas": grupo["sum"], "color": color})
    return pd.DataFrame(filas).sort_values("clase").reset_index(drop=True)


tabla_area = calcular_area_por_categoria(mapa_cambios, aoi)
tabla_area["porcentaje"] = 100 * tabla_area["hectareas"] / tabla_area["hectareas"].sum()

pd.set_option("display.float_format", lambda x: f"{x:,.1f}")
tabla_area[["categoria", "hectareas", "porcentaje"]]

### 4.2 Estadísticas de área por año (comparaciones consecutivas)

Aplica `calcular_area_por_categoria` (4.1) a cada uno de los 16 mapas de cambio año a año de la
Fase 3.6, produciendo una tabla larga (`periodo`, `categoria`, `hectareas`) que alimenta la
visualización de evolución interanual de la Fase 5.5.

In [ ]:
filas_anuales = []
for par_label, mapa_par in MAPAS_CAMBIO_ANUAL.items():
    tabla_par = calcular_area_por_categoria(mapa_par, aoi)
    tabla_par["periodo"] = par_label
    filas_anuales.append(tabla_par)

tabla_area_anual = pd.concat(filas_anuales, ignore_index=True)
tabla_area_anual[["periodo", "categoria", "hectareas"]]

### 4.3 Validación cruzada multiclase (matriz de confusión, OA, AA y Kappa)

No se cuenta con datos de verdad de campo, por lo que la validación se hace comparando el mapa de
cambios propio contra un producto de cobertura de suelo **independiente y ya publicado**:
[Dynamic World](https://dynamicworld.app/) (Google/WRI), que ofrece probabilidades de cobertura
(`built`, `trees`, `grass`, `crops`, `shrub_and_scrub`, `water`, …) casi en tiempo real desde 2015,
a 10 m de resolución. A partir de esas probabilidades se construye un mapa de cambios de
**referencia** con las mismas 5 categorías que nuestro mapa propio (Fase 3.5), y se comparan ambos
con las métricas estándar de evaluación de clasificaciones en teledetección — el mismo esquema que
se usa, por ejemplo, al validar clasificadores sobre el conjunto de referencia **Indian Pines**:

- **Matriz de confusión** (una fila/columna por categoría presente en la muestra).
- **OA (Overall Accuracy / exactitud global)**: proporción total de puntos de muestra coincidentes.
- **AA (Average Accuracy / exactitud promedio)**: promedio de la exactitud (sensibilidad) de cada
  categoría por separado — más exigente que la OA cuando hay categorías minoritarias, como suele
  ocurrir con "cambio en cuerpos de agua" frente a "sin cambio".
- **Índice Kappa de Cohen**: concordancia corregida por azar.

El muestreo es **aleatorio estratificado** (`stratifiedSample`, la misma técnica usada para separar
puntos de entrenamiento/prueba de forma balanceada en trabajos de clasificación de imágenes): se
toman puntos por igual de cada una de las 5 categorías, para que las clases minoritarias no queden
subrepresentadas.

**Nota importante sobre Dynamic World como referencia:** al usar el rango completo 2010→2026 para
la comparación principal, Dynamic World (disponible solo desde 2015) valida realmente el periodo
**2015→2026**, no 2010→2015. Es la mejor referencia independiente de acceso abierto disponible; se
declara esta limitación de forma explícita, tal como exige la buena práctica de validación en
teledetección.

In [ ]:
def obtener_prob_dw(banda, fecha_inicio, fecha_fin, aoi):
    """Promedio de una banda de probabilidad de Dynamic World (0-1) sobre un periodo y el AOI."""
    return (
        ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1")
        .filterBounds(aoi)
        .filterDate(fecha_inicio, fecha_fin)
        .select(banda)
        .median()
        .clip(aoi)
    )

# Dynamic World solo existe desde junio de 2015: para el extremo inicial (2010) se usa la ventana
# disponible más temprana dentro del periodo de estudio (ver nota metodológica arriba).
FECHA_DW_DISPONIBLE = "2015-06-01"
fecha_ini_dw = max(PERIODOS[f"{T_INICIAL_LABEL}-S1"][0], FECHA_DW_DISPONIBLE)

# Probabilidad de vegetación combinada (árboles + pastos + cultivos + arbustos) para el periodo
# inicial y final de la comparación principal; "built" y "water" se usan directamente.
BANDAS_VEGETACION_DW = ["trees", "grass", "crops", "shrub_and_scrub"]

def obtener_vegetacion_dw(fecha_inicio, fecha_fin, aoi):
    bandas = [obtener_prob_dw(b, fecha_inicio, fecha_fin, aoi) for b in BANDAS_VEGETACION_DW]
    return bandas[0].add(bandas[1]).add(bandas[2]).add(bandas[3]).rename("vegetacion")

veg_inicial_dw = obtener_vegetacion_dw(fecha_ini_dw, PERIODOS[f"{T_INICIAL_LABEL}-S2"][1], aoi)
veg_final_dw = obtener_vegetacion_dw(*PERIODOS[f"{T_FINAL_LABEL}-S1"], aoi)
built_inicial_dw = obtener_prob_dw("built", fecha_ini_dw, PERIODOS[f"{T_INICIAL_LABEL}-S2"][1], aoi)
built_final_dw = obtener_prob_dw("built", *PERIODOS[f"{T_FINAL_LABEL}-S1"], aoi)
water_inicial_dw = obtener_prob_dw("water", fecha_ini_dw, PERIODOS[f"{T_INICIAL_LABEL}-S2"][1], aoi)
water_final_dw = obtener_prob_dw("water", *PERIODOS[f"{T_FINAL_LABEL}-S1"], aoi)

delta_veg_dw = veg_final_dw.subtract(veg_inicial_dw)
delta_built_dw = built_final_dw.subtract(built_inicial_dw)
delta_water_dw = water_final_dw.subtract(water_inicial_dw)

# Umbral de cambio mínimo en probabilidad de Dynamic World (0-1) para considerarlo relevante.
UMBRAL_DW = 0.10

mapa_cambios_dw = (
    ee.Image(0)
    .where(delta_veg_dw.gt(UMBRAL_DW), 2)
    .where(delta_veg_dw.lt(-UMBRAL_DW), 1)
    .where(delta_built_dw.gt(UMBRAL_DW), 3)
    .where(delta_water_dw.abs().gt(UMBRAL_DW), 4)
    .rename("clase_cambio_dw")
)

print("Mapa de cambios de referencia (Dynamic World) construido con las mismas 5 categorías.")

Muestreo aleatorio estratificado y cálculo de las métricas de validación:

In [ ]:
from sklearn.metrics import confusion_matrix, cohen_kappa_score, classification_report

NOMBRES_CLASES = [CATEGORIAS_CAMBIO[c][0] for c in range(5)]

capas_comparacion = mapa_cambios.rename("propio").addBands(mapa_cambios_dw.rename("referencia"))

muestra = capas_comparacion.stratifiedSample(
    numPoints=80,          # puntos por categoría (muestreo balanceado; hasta 5 x 80 = 400 puntos)
    classBand="propio",
    region=aoi,
    scale=20,
    seed=42,
    geometries=False,
).getInfo()

registros = [f["properties"] for f in muestra["features"]]
df_muestra = pd.DataFrame(registros).dropna()

y_propio = df_muestra["propio"].astype(int)
y_referencia = df_muestra["referencia"].astype(int)

etiquetas_presentes = sorted(set(y_referencia) | set(y_propio))
matriz_confusion = confusion_matrix(y_referencia, y_propio, labels=etiquetas_presentes)

oa = (y_referencia == y_propio).mean()
kappa = cohen_kappa_score(y_referencia, y_propio)
exactitud_por_clase = matriz_confusion.diagonal() / matriz_confusion.sum(axis=1).clip(min=1)
aa = exactitud_por_clase.mean()

print(f"Tamaño de la muestra evaluada: {len(df_muestra)} puntos ({len(etiquetas_presentes)} categorías presentes)")
print(f"OA  (Overall Accuracy):  {oa:.1%}")
print(f"AA  (Average Accuracy):  {aa:.1%}")
print(f"Kappa de Cohen:          {kappa:.3f}")

print("\nReporte de clasificación (precisión / sensibilidad / F1 por categoría):")
print(classification_report(
    y_referencia, y_propio,
    labels=etiquetas_presentes,
    target_names=[NOMBRES_CLASES[c] for c in etiquetas_presentes],
    zero_division=0,
))

**Cómo interpretar el Kappa** (escala estándar de Landis & Koch, 1977, ampliamente citada en
estudios de exactitud temática en teledetección):

| Kappa | Interpretación |
|---|---|
| < 0.00 | Sin acuerdo |
| 0.00 – 0.20 | Leve |
| 0.21 – 0.40 | Aceptable |
| 0.41 – 0.60 | Moderado |
| 0.61 – 0.80 | Sustancial |
| 0.81 – 1.00 | Casi perfecto |

Matriz de confusión normalizada (proporción de cada categoría de referencia clasificada en cada
categoría propia):

In [ ]:
matriz_normalizada = matriz_confusion / matriz_confusion.sum(axis=1, keepdims=True).clip(min=1)
etiquetas_clase = [NOMBRES_CLASES[c] for c in etiquetas_presentes]

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(matriz_normalizada, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(etiquetas_clase)))
ax.set_yticks(range(len(etiquetas_clase)))
ax.set_xticklabels(etiquetas_clase, rotation=40, ha="right")
ax.set_yticklabels(etiquetas_clase)
ax.set_xlabel("Categoría — mapa propio")
ax.set_ylabel("Categoría — referencia (Dynamic World)")
ax.set_title(f"Matriz de confusión normalizada (OA={oa:.1%}, AA={aa:.1%}, Kappa={kappa:.3f})")

for i in range(len(etiquetas_clase)):
    for j in range(len(etiquetas_clase)):
        color_texto = "white" if matriz_normalizada[i, j] > 0.5 else "black"
        ax.text(j, i, f"{matriz_normalizada[i, j]:.0%}", ha="center", va="center", color=color_texto, fontsize=9)

fig.colorbar(im, ax=ax, label="Proporción dentro de cada categoría de referencia")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "matriz_confusion.png"), dpi=150)
plt.show()

### 4.4 Análisis de sensibilidad del umbral (robustez del método)

Se recalcula el porcentaje de área con cambios significativos usando distintos valores de *k*
(multiplicador de la desviación estándar) para verificar que la herramienta no depende de forma
crítica de un único valor arbitrario — otro argumento de confiabilidad para el proyecto.

In [ ]:
valores_k = [1.0, 1.25, 1.5, 1.75, 2.0]
resultados_sensibilidad = []

for k in valores_k:
    umb_ndvi_inf, umb_ndvi_sup = obtener_umbrales(deltas, "delta_NDVI", aoi, k=k)
    umb_ndbi_inf, umb_ndbi_sup = obtener_umbrales(deltas, "delta_NDBI", aoi, k=k)

    mapa_k = (
        ee.Image(0)
        .where(deltas.select("delta_NDVI").gt(umb_ndvi_sup), 2)
        .where(deltas.select("delta_NDVI").lt(umb_ndvi_inf), 1)
        .where(deltas.select("delta_NDBI").gt(umb_ndbi_sup), 3)
    )

    area_cambio_km2 = (
        ee.Image.pixelArea().divide(1e6)
        .updateMask(mapa_k.gt(0))
        .reduceRegion(reducer=ee.Reducer.sum(), geometry=aoi, scale=20, bestEffort=True, maxPixels=1e9)
        .getInfo()
    )
    km2 = area_cambio_km2.get("area", 0)
    resultados_sensibilidad.append({"k": k, "area_cambio_km2": km2, "pct_area_total": 100 * km2 / area_km2})

tabla_sensibilidad = pd.DataFrame(resultados_sensibilidad)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(tabla_sensibilidad["k"], tabla_sensibilidad["pct_area_total"], marker="o", color="#2166ac")
ax.set_xlabel("k (umbral = media ± k·desviación estándar)")
ax.set_ylabel("% del AOI clasificado como cambio")
ax.set_title("Análisis de sensibilidad del umbral estadístico")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "sensibilidad_umbral.png"), dpi=150)
plt.show()

tabla_sensibilidad

## Fase 5. Visualización de los resultados

**Objetivo específico:** *Generar visualizaciones de los cambios detectados, empleando librerías
de representación gráfica, permitiendo la interpretación de los cambios detectados.*

**Qué se hizo:** siete productos visuales complementarios — un mapa interactivo, una comparación
estática antes/después/cambios, un panel con los 16 mapas de cambio año a año, una serie de tiempo
de índices, un gráfico de evolución interanual del área por categoría, un gráfico de barras de la
comparación principal, y la exportación de todos los productos.
**Por qué varias visualizaciones y no una sola:** cada una responde una pregunta distinta —
¿dónde? (mapas), ¿cuánto? (barras/áreas), ¿cuándo? (series de tiempo, panel año a año) — necesarias
en conjunto para interpretar un fenómeno espacio-temporal como el cambio de cobertura terrestre.
**Herramientas:** `geemap` (mapas interactivos), `matplotlib`/`pandas` (gráficos estáticos).
**Qué arroja:** figuras PNG guardadas en `CARPETA_RESULTADOS`, listas para incluir en el documento
de tesis, más las tablas y el mapa de cambios exportados como GeoTIFF/CSV.

### 5.1 Mapa interactivo comparativo (antes / después / cambios)

In [ ]:
mapa_final = geemap.Map(center=AMB_CENTRO, zoom=11, basemap="ROADMAP")
mapa_final.addLayer(imagen_inicial, vis_rgb, f"Color verdadero {T_INICIAL_LABEL}", shown=False)
mapa_final.addLayer(imagen_final, vis_rgb, f"Color verdadero {T_FINAL_LABEL}", shown=True)
mapa_final.addLayer(
    preparar_para_mapa(mapa_cambios),
    VIS_CAMBIOS,
    f"Mapa de cambios {T_INICIAL_LABEL} → {T_FINAL_LABEL}",
)
mapa_final.addLayer(ee.Image().paint(aoi, 0, 2), {"palette": ["black"]}, "Límite del AOI (AMB)")
mapa_final.add_legend(
    title="Categoría de cambio (0 se deja transparente)",
    legend_dict={nombre: color for nombre, color in CATEGORIAS_CAMBIO.values()},
)
mapa_final.addLayerControl()
mapa_final

### 5.2 Comparación visual estática (antes / después / cambios) — comparación principal

Reutiliza `obtener_imagen_ee` (definida en la Fase 1.7) para descargar las miniaturas como arreglos
numpy y componer la figura comparativa del periodo completo de estudio (2010 → 2026).

In [ ]:
img_inicial_arr = np.array(obtener_imagen_ee(imagen_inicial, vis_rgb))
img_final_arr = np.array(obtener_imagen_ee(imagen_final, vis_rgb))
img_cambios_arr = np.array(obtener_imagen_ee(preparar_para_mapa(mapa_cambios), VIS_CAMBIOS))

fig, ejes = plt.subplots(1, 3, figsize=(16, 5.5))

ejes[0].imshow(img_inicial_arr)
ejes[0].set_title(f"Área Metropolitana de Bucaramanga — {T_INICIAL_LABEL}")
ejes[0].axis("off")

ejes[1].imshow(img_final_arr)
ejes[1].set_title(f"Área Metropolitana de Bucaramanga — {T_FINAL_LABEL}")
ejes[1].axis("off")

ejes[2].imshow(img_cambios_arr)
ejes[2].set_title(f"Cambios detectados ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
ejes[2].axis("off")

parches_leyenda = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=color, markersize=12, label=nombre)
    for nombre, color in CATEGORIAS_CAMBIO.values()
]
fig.legend(handles=parches_leyenda, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "comparacion_antes_despues_cambios.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.3 Panel de mapas de cambio año a año

Un mosaico de mapas —uno por cada comparación consecutiva de la Fase 3.6— para **comparar
visualmente cómo cambió el AMB año a año**, en vez de ver solo el resultado acumulado de todo el
periodo de estudio. Es la visualización central para responder "¿qué cambió, y en qué año?".

In [ ]:
n_pares = len(MAPAS_CAMBIO_ANUAL)
n_columnas = min(4, n_pares)
n_filas = -(-n_pares // n_columnas)  # división entera hacia arriba

fig, ejes = plt.subplots(n_filas, n_columnas, figsize=(4.2 * n_columnas, 4.6 * n_filas))
ejes = np.atleast_1d(ejes).flatten()

for eje, (par_label, mapa_par) in zip(ejes, MAPAS_CAMBIO_ANUAL.items()):
    arreglo = np.array(obtener_imagen_ee(preparar_para_mapa(mapa_par), VIS_CAMBIOS, dimensiones=512))
    eje.imshow(arreglo)
    eje.set_title(par_label, fontsize=11)
    eje.axis("off")

for eje_sobrante in ejes[n_pares:]:
    eje_sobrante.axis("off")

parches_leyenda = [
    plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=color, markersize=12, label=nombre)
    for nombre, color in CATEGORIAS_CAMBIO.values()
]
fig.legend(handles=parches_leyenda, loc="lower center", ncol=3, bbox_to_anchor=(0.5, -0.01), frameon=False)
fig.suptitle("Mapas de cambio año a año — Área Metropolitana de Bucaramanga", fontsize=14, y=1.01)

plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "mapas_cambio_anual.png"), dpi=150, bbox_inches="tight")
plt.show()

### 5.4 Serie de tiempo de NDVI y NDBI promedio en el AMB (monitoreo multitemporal)

In [ ]:
registros_series = []
for etiqueta in ETIQUETAS_ANIOS:
    promedios = COMPOSITES[etiqueta].select(["NDVI", "NDBI"]).reduceRegion(
        reducer=ee.Reducer.mean(), geometry=aoi, scale=30, bestEffort=True, maxPixels=1e9,
    ).getInfo()
    registros_series.append({"periodo": etiqueta, "NDVI_promedio": promedios["NDVI"], "NDBI_promedio": promedios["NDBI"]})

serie_tiempo = pd.DataFrame(registros_series)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDVI_promedio"], marker="o", color="#1a9850", label="NDVI promedio (vegetación)")
ax.plot(serie_tiempo["periodo"], serie_tiempo["NDBI_promedio"], marker="s", color="#fdae61", label="NDBI promedio (suelo construido)")
ax.axvline(x="2016", color="grey", linestyle="--", alpha=0.6)
ax.text(0.5, 0.02, "← Landsat (30 m)   |   Sentinel-2 (10 m) →", transform=ax.transAxes, ha="center", fontsize=9, color="grey")
ax.set_xlabel("Año")
ax.set_ylabel("Valor promedio del índice")
ax.set_title("Evolución multitemporal de NDVI y NDBI en el AMB (2010-2026)")
ax.legend()
ax.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "serie_tiempo_ndvi_ndbi.png"), dpi=150)
plt.show()

serie_tiempo

### 5.5 Evolución del área por categoría a través de los años

A partir de `tabla_area_anual` (Fase 4.2), se grafica cómo cambió el área (hectáreas) de cada
categoría en cada intervalo año a año — la comparación interanual que permite identificar
tendencias (p. ej. si la expansión urbana se acelera o si la pérdida de vegetación es constante),
en lugar de un único número acumulado para todo el periodo de estudio.

In [ ]:
tabla_pivote = (
    tabla_area_anual[tabla_area_anual["clase"] != 0]
    .pivot(index="periodo", columns="categoria", values="hectareas")
    .reindex([f"{a}→{b}" for a, b in PARES_CONSECUTIVOS])
    .fillna(0)
)

colores_categoria = {nombre: color for nombre, color in CATEGORIAS_CAMBIO.values() if nombre != CATEGORIAS_CAMBIO[0][0]}

fig, ax = plt.subplots(figsize=(13, 5.5))
tabla_pivote.plot(kind="bar", ax=ax, color=[colores_categoria.get(col, "#999999") for col in tabla_pivote.columns])
ax.set_xlabel("Comparación año a año")
ax.set_ylabel("Área (hectáreas)")
ax.set_title("Evolución interanual del área por categoría de cambio — AMB")
ax.legend(title="Categoría", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.grid(axis="y", alpha=0.3)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "evolucion_area_anual.png"), dpi=150, bbox_inches="tight")
plt.show()

tabla_pivote

### 5.6 Gráfico de área por categoría de cambio (comparación principal)

In [ ]:
tabla_area_graf = tabla_area[tabla_area["clase"] != 0].sort_values("hectareas", ascending=True)

fig, ax = plt.subplots(figsize=(8, 4.5))
barras = ax.barh(tabla_area_graf["categoria"], tabla_area_graf["hectareas"], color=tabla_area_graf["color"])
ax.set_xlabel("Área (hectáreas)")
ax.set_title(f"Área por categoría de cambio — AMB ({T_INICIAL_LABEL} → {T_FINAL_LABEL})")
for barra, valor in zip(barras, tabla_area_graf["hectareas"]):
    ax.text(valor, barra.get_y() + barra.get_height() / 2, f" {valor:,.0f} ha", va="center")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(CARPETA_RESULTADOS, "area_por_categoria.png"), dpi=150)
plt.show()

### 5.7 Exportar resultados

Se guardan/descargan los productos finales:
1. **Tablas de estadísticas**: `estadisticas_cambio_principal.csv` (Fase 4.1) y
   `estadisticas_cambio_anual.csv` (Fase 4.2, año a año).
2. **Figuras** (PNG) generadas en 4.3, 4.4, 5.2, 5.3, 5.5 y 5.6 — ya guardadas en `CARPETA_RESULTADOS`.
3. **Mapa de cambios clasificado de la comparación principal** (GeoTIFF) — exportado a Google Drive
   mediante una tarea de Earth Engine (puede tardar algunos minutos; revisa el progreso en la
   pestaña *Tasks* de https://code.earthengine.google.com/). Los mapas año a año ya quedaron
   guardados como imagen (PNG) en el panel de la Fase 5.3; si necesitas cada uno como GeoTIFF por
   separado para un SIG, repite este mismo patrón de exportación dentro de un bucle sobre
   `MAPAS_CAMBIO_ANUAL`.

**Nota sobre la resolución de exportación:** la comparación principal combina un año Landsat
(30 m nativos) con un año Sentinel-2 (10 m nativos); se exporta a 10 m por consistencia con el resto
del notebook, pero la porción de la información que proviene del año Landsat conserva su resolución
efectiva original de 30 m (Earth Engine remuestrea la grilla, no inventa detalle nuevo). Es un límite
inherente a combinar sensores, no un error de la herramienta — documentarlo es parte de la
validación de confiabilidad de la Fase 4.

In [ ]:
ruta_csv = os.path.join(CARPETA_RESULTADOS, "estadisticas_cambio_principal.csv")
tabla_area.to_csv(ruta_csv, index=False)
print(f"Tabla de estadísticas (comparación principal) guardada en: {ruta_csv}")

ruta_csv_anual = os.path.join(CARPETA_RESULTADOS, "estadisticas_cambio_anual.csv")
tabla_area_anual.to_csv(ruta_csv_anual, index=False)
print(f"Tabla de estadísticas (año a año) guardada en: {ruta_csv_anual}")

tarea_exportacion = ee.batch.Export.image.toDrive(
    image=mapa_cambios.toByte(),
    description="mapa_cambios_AMB",
    folder="deteccion_cambios_bucaramanga",
    fileNamePrefix=f"mapa_cambios_{T_INICIAL_LABEL}_{T_FINAL_LABEL}",
    region=aoi,
    scale=10,
    maxPixels=1e9,
)
tarea_exportacion.start()
print("Tarea de exportación del mapa de cambios principal (GeoTIFF) iniciada.")
print("Revisa su progreso en la pestaña 'Tasks' de https://code.earthengine.google.com/")

from google.colab import files
files.download(ruta_csv)
files.download(ruta_csv_anual)

## Conclusiones y próximos pasos

Este notebook cubrió, fase a fase, los cinco objetivos específicos del proyecto para el Área
Metropolitana de Bucaramanga, con un diseño pensado para sostener un trabajo de investigación
académica (tesis de grado):

1. **Recopilación** — 17 años (2010-2026), 2 ventanas estacionales cada uno (34 imágenes
   recopiladas en total), combinando Landsat (2010-2016) y Sentinel-2 (2017-2026) de acceso
   abierto, con selección automática de sensor por año y armonización de bandas.
2. **Preprocesamiento** — Enmascarado de nubes específico por sensor (SCL para Sentinel-2,
   QA_PIXEL para Landsat), compuestos anuales robustos a partir de 2 ventanas estacionales, e
   índices espectrales (NDVI, NDBI, NDWI) calculados de forma sensor-agnóstica.
3. **Sistematización** — Diferencias de índices con umbral estadístico reproducible, empaquetadas
   en funciones reutilizables (`calcular_deltas`, `obtener_umbrales`, `clasificar_cambios`) y
   aplicadas tanto a la comparación principal (2010→2026) como a **cada uno de los 16 pares de
   años consecutivos**.
4. **Evaluación** — Estadísticas de área (principal y año a año), validación cruzada **multiclase**
   con Dynamic World (matriz de confusión, **OA, AA y Kappa**, con su escala de interpretación de
   Landis & Koch) y análisis de sensibilidad del umbral — el sustento cuantitativo de confiabilidad
   del proyecto.
5. **Visualización** — Mapa interactivo, comparación antes/después, panel de mapas de cambio año a
   año, serie de tiempo, evolución interanual del área por categoría y exportación de resultados.

**Limitaciones metodológicas declaradas** (buena práctica académica: documentarlas, no ocultarlas):
- El par 2016→2017 mezcla sensores (Landsat 30 m → Sentinel-2 10 m); interpretarlo con cautela.
- La validación con Dynamic World (Fase 4.3) solo cubre datos desde 2015, no desde 2010.
- Los años 2010-2016 tienen menor resolución espacial (30 m) que 2017-2026 (10 m).

**Posibles extensiones:**
- Incorporar imágenes de mayor resolución (p. ej. PlanetScope) para detectar cambios más finos.
- Aumentar la densidad temporal (trimestral) modificando `VENTANAS_ESTACIONALES` (Fase 1.3).
- Sustituir la clasificación basada en umbrales por un clasificador supervisado (Random Forest /
  redes neuronales) entrenado con puntos de verdad de campo, si llegan a estar disponibles — la
  validación de la Fase 4.3 (OA/AA/Kappa/matriz de confusión) seguiría siendo aplicable sin cambios.
- Aplicar la misma metodología a otras zonas cambiando únicamente `aoi_bbox` y `MUNICIPIOS_AMB` (Fase 1).

## Solución de problemas

- **`EEException: Not signed up for Earth Engine` o error de autenticación**: crea/activa tu cuenta en
  https://code.earthengine.google.com/register y verifica que `EE_PROJECT_ID` (Fase 0.2) sea el ID
  correcto de tu proyecto de Google Cloud.
- **`0 imágenes encontradas` en algún año/ventana (Fase 1.5)**: es más probable en años Landsat
  (2010-2016, revisita cada 16 días vs. 5 días de Sentinel-2); amplía la ventana estacional
  correspondiente en `VENTANAS_ESTACIONALES` (1.3) o sube `MAX_NUBES_ESCENA` (1.3).
- **`Computation timed out` o `Too many pixels in the region`**: reduce el parámetro `scale` en las
  llamadas a `reduceRegion` (por ejemplo de 10 a 20 o 30 m) o reduce el tamaño del AOI.
- **La celda 1.2 imprime "No fue posible refinar el AOI"**: no es un error crítico — el notebook
  sigue funcionando con el rectángulo delimitador definido en la celda 1.1.
- **La imagen se ve "rota", cortada en diagonal, o con partes del AOI en blanco (Fase 1.6/1.7)**:
  es normal en una sola escena, que no siempre cubre el AOI completo (borde de la franja de
  barrido del satélite). Por eso estas celdas usan `obtener_mosaico_periodo` (combina varias
  escenas) en vez de una sola imagen; si el hueco persiste, esa ventana tiene muy pocas escenas
  disponibles — amplía su rango de fechas (1.3) o sube `MAX_NUBES_ESCENA`.
- **Las Fases 1.5, 1.7, 3.6, 4.2 y 5.3 tardan varios minutos**: es normal — con 17 años x 2 ventanas
  el notebook recalcula la metodología completa muchas veces. Si necesitas una ejecución más rápida
  (a costa de menos detalle histórico), reduce temporalmente `AÑOS_ANALISIS` (Fase 1.3).
- **El Kappa/OA/AA de la Fase 4.3 salen bajos**: es esperable cierto desacuerdo, ya que Dynamic
  World y el método propio usan sensores/criterios distintos, y para 2010 la comparación usa la
  ventana de Dynamic World disponible más temprana (2015); revisa `UMBRAL_DW` y `K_UMBRAL` — el
  análisis de sensibilidad (4.4) ayuda a elegir un valor de `K_UMBRAL` más estable.
- **`classification_report` o la matriz de confusión de la Fase 4.3 no incluyen las 5 categorías**:
  ocurre si alguna categoría (p. ej. "cambio en cuerpos de agua") no aparece en la muestra aleatoria
  del AOI; el código ya filtra dinámicamente por `etiquetas_presentes`, así que no es un error —
  simplemente esa categoría es poco frecuente en el periodo/zona analizados.
- **`KeyError` en una banda al mezclar Landsat/Sentinel-2**: si modificas `config_sensor` (Fase 1.4),
  recuerda mantener siempre las 5 bandas comunes de salida (`BLUE, GREEN, RED, NIR, SWIR1`) — el
  resto del notebook (índices, deltas, clasificación) depende de esos nombres, no de los nombres
  nativos de cada sensor.
- **En los mapas interactivos aparecen teselas con "Access blocked... osm.wiki/Blocked" o
  "API KEY REQUIRED"**: no es un error del notebook — son los proveedores externos de mapa base
  (OpenStreetMap bloquea el acceso automatizado; CartoDB movió sus estilos gratuitos detrás de una
  API key en 2023-2024). Tus capas de Earth Engine (imágenes, NDVI, mapa de cambios) se siguen
  renderizando bien encima, sin depender de esto. Ya se usa `basemap="ROADMAP"` (Google Maps, sin
  API key) en los 4 mapas del notebook; si tampoco carga, prueba `basemap="HYBRID"` o
  `basemap="TERRAIN"` en esas mismas celdas, o vuelve a ejecutar la celda (a veces son teselas
  cacheadas de una ejecución anterior con otro basemap).